In [1]:

import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. Locate and load the processed dataset
candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (path for path in candidates if (path / "data" / "processed" / "happiness_unified.csv").is_file()),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not locate data/processed/happiness_unified.csv")

data_path = project_root / "data" / "processed" / "happiness_unified.csv"
df = pd.read_csv(data_path)

# 2. Select the 2018–2019 records
modeling_df = df[df["year"].isin([2018, 2019])].copy()

# 3. Define predictors and target
features = [
    "gdp",
    "social_support",
    "health",
    "freedom",
    "generosity",
    "corruption"
]

target = "happiness_score"

X = modeling_df[features]
y = modeling_df[target]

# 4. Split data before fitting any preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42
)

# 5. Define preprocessing (not fitted yet)
preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Compact validation
print("DATASET:", data_path.name)
print("Selected years:", sorted(modeling_df["year"].unique()))
print("Modeling dataset:", modeling_df.shape)
print("Predictors:", features)
print("Target:", target)
print("Missing predictor values:", X.isnull().sum().sum())
print("Missing target values:", y.isnull().sum())
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)


DATASET: happiness_unified.csv
Selected years: [np.int64(2018), np.int64(2019)]
Modeling dataset: (312, 11)
Predictors: ['gdp', 'social_support', 'health', 'freedom', 'generosity', 'corruption']
Target: happiness_score
Missing predictor values: 1
Missing target values: 0
Training set: (218, 6)
Test set: (94, 6)


In [2]:

# Validate feature selection and target leakage prevention

excluded_variables = ["happiness_rank", "country", "year", "region"]

assert target not in features, "Target leakage: target included in predictors"
assert not any(col in features for col in excluded_variables), \
    "An excluded variable was included in the predictors"

assert X_train.columns.tolist() == features
assert X_test.columns.tolist() == features
assert y_train.name == target and y_test.name == target

# Confirm preprocessing has not been fitted using the test set
assert not hasattr(preprocessor.named_steps["imputer"], "statistics_")

print("FEATURE ENGINEERING VALIDATION")
print("Predictors:", len(features))
print("Excluded variables:", excluded_variables)
print("Target included among predictors:", target in features)
print("Missing values before preprocessing:", X_train.isnull().sum().sum())
print("Preprocessor fitted:", hasattr(preprocessor.named_steps["imputer"], "statistics_"))
print("Train/test separation: validated")


FEATURE ENGINEERING VALIDATION
Predictors: 6
Excluded variables: ['happiness_rank', 'country', 'year', 'region']
Target included among predictors: False
Missing values before preprocessing: 0
Preprocessor fitted: False
Train/test separation: validated


In [3]:

# 1. Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 2. Recover column names for inspection
X_train_processed = pd.DataFrame(
    X_train_processed,
    columns=features,
    index=X_train.index
)

X_test_processed = pd.DataFrame(
    X_test_processed,
    columns=features,
    index=X_test.index
)

# 3. Validate preprocessing
assert X_train_processed.isnull().sum().sum() == 0
assert X_test_processed.isnull().sum().sum() == 0
assert X_train_processed.shape == X_train.shape
assert X_test_processed.shape == X_test.shape

# 4. Validate that the imputer learned only from training data
print("PREPROCESSING RESULTS")
print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)
print("Missing values after preprocessing:",
      X_train_processed.isnull().sum().sum() +
      X_test_processed.isnull().sum().sum())
print("Imputation statistics:",
      dict(zip(features, preprocessor.named_steps["imputer"].statistics_.round(4))))
print("Training means after scaling:",
      X_train_processed.mean().round(4).to_dict())
print("Training standard deviations:",
      X_train_processed.std(ddof=0).round(4).to_dict())

print("\nTARGET LEAKAGE CHECK")
print("Target:", target)
print("Target excluded from predictors:", target not in features)
print("Excluded rank variable:", "happiness_rank" not in features)
print("Preprocessing fitted on training set only: True")
print("Test set used only for transformation: True")


PREPROCESSING RESULTS
Training shape: (218, 6)
Testing shape: (94, 6)
Missing values after preprocessing: 0
Imputation statistics: {'gdp': np.float64(0.9325), 'social_support': np.float64(1.255), 'health': np.float64(0.6885), 'freedom': np.float64(0.4545), 'generosity': np.float64(0.1765), 'corruption': np.float64(0.0855)}
Training means after scaling: {'gdp': -0.0, 'social_support': 0.0, 'health': -0.0, 'freedom': 0.0, 'generosity': 0.0, 'corruption': 0.0}
Training standard deviations: {'gdp': 1.0, 'social_support': 1.0, 'health': 1.0, 'freedom': 1.0, 'generosity': 1.0, 'corruption': 1.0}

TARGET LEAKAGE CHECK
Target: happiness_score
Target excluded from predictors: True
Excluded rank variable: True
Preprocessing fitted on training set only: True
Test set used only for transformation: True


In [4]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import joblib

# 1. Build the complete machine learning pipeline
model_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

# 2. Train using only the training set
model_pipeline.fit(X_train, y_train)

# 3. Generate predictions on unseen test data
y_pred = model_pipeline.predict(X_test)

# 4. Evaluate model performance
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print("MODEL: LINEAR REGRESSION")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("\nEVALUATION METRICS")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# 5. Save the complete pipeline
model_dir = project_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "model.pkl"
joblib.dump(model_pipeline, model_path)

print("\nMODEL SAVED")
print("Path:", model_path)
print("Pipeline steps:", list(model_pipeline.named_steps.keys()))


MODEL: LINEAR REGRESSION
Training records: 218
Testing records: 94

EVALUATION METRICS
MAE:  0.4124
RMSE: 0.5207
R²:   0.7651

MODEL SAVED
Path: c:\Users\USER\Desktop\ETL-Workshop-3\models\model.pkl
Pipeline steps: ['imputer', 'scaler', 'regressor']


In [6]:

# Reload the saved model
loaded_model = joblib.load(model_path)

# Generate predictions using the reloaded pipeline
sample_predictions = loaded_model.predict(X_test.iloc[:5])

# Validate the saved model
assert model_path.is_file()
assert len(sample_predictions) == 5
assert all(pd.notna(sample_predictions))
assert list(loaded_model.named_steps.keys()) == [
    "imputer", "scaler", "regressor"
]

print("MODEL VALIDATION")
print("Model file exists:", model_path.is_file())
print("Model successfully reloaded:", True)
print("Sample predictions generated:", len(sample_predictions))
print("Predictions contain valid values:", all(pd.notna(sample_predictions)))
print("Pipeline structure:", list(loaded_model.named_steps.keys()))

print("\nSAMPLE PREDICTIONS")
print(pd.DataFrame({
    "Actual": y_test.iloc[:5].values,
    "Predicted": sample_predictions.round(4)
}))


MODEL VALIDATION
Model file exists: True
Model successfully reloaded: True
Sample predictions generated: 5
Predictions contain valid values: True
Pipeline structure: ['imputer', 'scaler', 'regressor']

SAMPLE PREDICTIONS
   Actual  Predicted
0   5.523     5.4695
1   7.272     6.9600
2   5.835     6.0689
3   5.762     5.8123
4   6.441     6.1704
